# 04 — Pipeline TTS

ElevenLabs, OpenAI, Cartesia, LMNT, Rime.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? 'unknown';
  console.log(`getpatter ${v} on ${typeof Deno !== 'undefined' ? `Deno ${Deno.version.deno}` : `Node ${process.version}`}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  if ((p as any)._mode !== 'local') throw new Error(`expected local, got ${(p as any)._mode}`);
  console.log(`mode = ${(p as any)._mode}`);
});


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  const p = new Patter({ apiKey: 'pt_test_xxx' });
  if ((p as any)._mode !== 'cloud') throw new Error(`expected cloud, got ${(p as any)._mode}`);
  console.log(`mode = ${(p as any)._mode}; apiKey = ${p.apiKey.slice(0, 8)}...`);
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  console.log(`realtime agent → ${rt.constructor.name}`);
  console.log(`convai agent   → ${cv.constructor.name}`);
  console.log(`pipeline agent → ${pl.constructor.name}`);
});


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.
